In [8]:
%%writefile bfs.cpp
#include <omp.h>
#include <iostream>
#include <vector>
#include <queue>

using namespace std;

// ---------------- BFS ----------------
void parallelBFS(vector<vector<int>> &graph, int start) {
    int n = graph.size();
    vector<bool> visited(n, false);
    queue<int> q;

    visited[start] = true;
    q.push(start);

    cout << "\nBFS Traversal: ";

    while (!q.empty()) {
        int size = q.size();
        vector<int> level;

        for (int i = 0; i < size; i++) {
            int node = q.front();
            q.pop();
            cout << node << " ";
            level.push_back(node);
        }

        #pragma omp parallel for
        for (int i = 0; i < level.size(); i++) {
            int u = level[i];

            for (int v : graph[u]) {
                if (!visited[v]) {
                    #pragma omp critical
                    {
                        if (!visited[v]) {
                            visited[v] = true;
                            q.push(v);
                        }
                    }
                }
            }
        }
    }

    cout << endl;
}

// ---------------- DFS ----------------
void dfsUtil(int node, vector<vector<int>> &graph, vector<bool> &visited) {

    // First check (outside critical)
    if (visited[node]) return;

    #pragma omp critical
    {
        if (!visited[node]) {
            visited[node] = true;
            cout << node << " ";
        }
    }

    #pragma omp parallel for
    for (int i = 0; i < graph[node].size(); i++) {
        int v = graph[node][i];
        if (!visited[v]) {
            dfsUtil(v, graph, visited);
        }
    }
}

void parallelDFS(vector<vector<int>> &graph, int start) {
    int n = graph.size();
    vector<bool> visited(n, false);

    cout << "\nDFS Traversal: ";
    dfsUtil(start, graph, visited);
    cout << endl;
}

// ---------------- MAIN ----------------
int main() {
    int n, e;

    cout << "Enter number of vertices: ";
    cin >> n;

    cout << "Enter number of edges: ";
    cin >> e;

    vector<vector<int>> graph(n);

    cout << "Enter edges (u v):\n";
    for (int i = 0; i < e; i++) {
        int u, v;
        cin >> u >> v;
        graph[u].push_back(v);
        graph[v].push_back(u); // undirected graph
    }

    int start, choice;

    cout << "Enter starting vertex: ";
    cin >> start;

    do {
        cout << "\n===== MENU =====\n";
        cout << "1. Parallel BFS\n";
        cout << "2. Parallel DFS\n";
        cout << "3. Exit\n";
        cout << "Enter choice: ";
        cin >> choice;

        switch(choice) {
            case 1:
                parallelBFS(graph, start);
                break;

            case 2:
                parallelDFS(graph, start);
                break;

            case 3:
                cout << "Exiting...\n";
                break;

            default:
                cout << "Invalid choice!\n";
        }

    } while(choice != 3);

    return 0;
}



Overwriting bfs.cpp


In [9]:
!g++ -fopenmp bfs.cpp -o bfs

In [12]:
!echo "5 4 0 1 0 2 1 3 1 4 0 1 3" | ./bfs

Enter number of vertices: Enter number of edges: Enter edges (u v):
Enter starting vertex: 
===== MENU =====
1. Parallel BFS
2. Parallel DFS
3. Exit
Enter choice: 
BFS Traversal: 0 1 2 3 4 

===== MENU =====
1. Parallel BFS
2. Parallel DFS
3. Exit
Enter choice: Exiting...


In [13]:
!echo -e "5\n4\n0 1\n0 2\n1 3\n1 4\n0\n1\n3" | ./bfs

Enter number of vertices: Enter number of edges: Enter edges (u v):
Enter starting vertex: 
===== MENU =====
1. Parallel BFS
2. Parallel DFS
3. Exit
Enter choice: 
BFS Traversal: 0 1 2 3 4 

===== MENU =====
1. Parallel BFS
2. Parallel DFS
3. Exit
Enter choice: Exiting...


In [14]:
!echo -e "5\n4\n0 1\n0 2\n1 3\n1 4\n0\n2\n3" | ./bfs

Enter number of vertices: Enter number of edges: Enter edges (u v):
Enter starting vertex: 
===== MENU =====
1. Parallel BFS
2. Parallel DFS
3. Exit
Enter choice: 
DFS Traversal: 0 2 1 3 4 

===== MENU =====
1. Parallel BFS
2. Parallel DFS
3. Exit
Enter choice: Exiting...
